In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# =========================
# 1. LOAD DATASETS
# =========================
books_path = "/content/drive/My Drive/EADA/EADA_DeepLearning/Project/books.csv"
clean_path = "/content/drive/My Drive/EADA/EADA_DeepLearning/Project/cleanver2_books_dataset.csv"

books_df = pd.read_csv(books_path)
clean_df = pd.read_csv(clean_path)

In [3]:
# =========================
# 2. NORMALIZATION FUNCTION
# =========================
def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

# Apply normalization to matching fields
books_df["isbn13"] = books_df["isbn13"].apply(normalize_text)
books_df["isbn10"] = books_df["isbn10"].apply(normalize_text)
books_df["title"] = books_df["title"].apply(normalize_text)
books_df["subtitle"] = books_df["subtitle"].apply(normalize_text)

clean_df["isbn_13"] = clean_df["isbn_13"].apply(normalize_text)
clean_df["isbn_10"] = clean_df["isbn_10"].apply(normalize_text)
clean_df["title"] = clean_df["title"].apply(normalize_text)
clean_df["subtitle"] = clean_df["subtitle"].apply(normalize_text)

In [4]:
# =========================
# 3. CREATE MATCHING KEYS
# =========================
books_df["key"] = (
    books_df["isbn13"] + "|" +
    books_df["isbn10"] + "|" +
    books_df["title"] + "|" +
    books_df["subtitle"]
)

clean_df["key"] = (
    clean_df["isbn_13"] + "|" +
    clean_df["isbn_10"] + "|" +
    clean_df["title"] + "|" +
    clean_df["subtitle"]
)

In [5]:
# =========================
# 4. IDENTIFY COMMON BOOKS
# =========================
common_books = books_df[
    books_df["key"].isin(clean_df["key"])
].copy()

print("Total overlapping books:", len(common_books))

Total overlapping books: 13


In [9]:
!pip install rapidfuzz

In [10]:
# --------------
# Priority 1: ISBN match
isbn_match = books_df["isbn13"].isin(clean_df["isbn_13"]) | \
             books_df["isbn10"].isin(clean_df["isbn_10"])

# Priority 2: Fuzzy title match (optional)
from rapidfuzz import fuzz

def fuzzy_match(row, clean_titles, threshold=90):
    return any(fuzz.ratio(row["title"], t) > threshold for t in clean_titles)

clean_titles = clean_df["title"].tolist()
books_df["fuzzy_match"] = books_df.apply(lambda r: fuzzy_match(r, clean_titles), axis=1)

# Combine logic
books_df["is_common"] = isbn_match | books_df["fuzzy_match"]
# --------------

In [11]:
# =========================
# 5. SHOW 13 SAMPLE MATCHES
# =========================
sample_13 = common_books[[
    "title", "subtitle", "isbn13", "isbn10"
]].head(13)

print("\nSample of 13 overlapping books:")
print(sample_13)


Sample of 13 overlapping books:
                                              title  \
300                                      good omens   
350                                              v.   
401                             the light fantastic   
1434                          dreams from my father   
2981                                the waste lands   
3006                       the drawing of the three   
3238                               heart of the sea   
3639                                       the firm   
3865    the seven habits of highly effective people   
3886                  principle centered leadership   
3899  daily reflections for highly effective people   
4508            harry potter and the goblet of fire   
5898                                angels & demons   

                                               subtitle         isbn13  \
300   the nice and accurate prophecies of agnes nutt...  9780060853983   
350                                             

In [13]:
# =========================
# 5. GET BOOKS NOT IN CLEAN
# =========================

missing_books = books_df[~books_df["is_common"]].copy()

print("Books to be added:", len(missing_books))


# =========================
# 6. ALIGN TO CLEAN SCHEMA
# =========================

aligned_missing = pd.DataFrame(columns=clean_df.columns)

mapping = {
    "isbn_13": "isbn13",
    "isbn_10": "isbn10",
    "title": "title",
    "subtitle": "subtitle",
    "authors": "authors",
    "categories": "categories",
    "thumbnail": "thumbnail",
    "description": "description",
    "average_rating": "average_rating",
    "ratings_count": "ratings_count",
    "page_count": "num_pages",
    "published_date": "published_year"
}

for col in clean_df.columns:
    if col in mapping and mapping[col] in missing_books.columns:
        aligned_missing[col] = missing_books[mapping[col]]
    else:
        aligned_missing[col] = None

Books to be added: 6557


In [14]:
# =========================
# 7. CREATE FINAL DF
# =========================

final_df = pd.concat([clean_df, aligned_missing], ignore_index=True)

print("Final dataset size:", len(final_df))

Final dataset size: 19054


/tmp/ipykernel_2494/1673337787.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([clean_df, aligned_missing], ignore_index=True)


In [15]:


# =========================
# 8. SAVE MERGED DATASET
# =========================

output_path = "/content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_dataset.csv"

final_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /content/drive/My Drive/EADA/EADA_DeepLearning/Project/merged_dataset.csv
